# Reproduce the optional saved analyses

The main numbered notebooks read the completed outputs. This notebook describes how to rerun the existing empirical-prior, support-bound and classifier diagnostics on copied archived inputs. It does not run them automatically. No NPE fit is needed for prior reweighting or bound sensitivity.

Use a separate research environment matching the saved model requirements before deserializing checkpoints. The empirical-prior calculation needs NumPy, pandas, SciPy, matplotlib, scikit-learn and threadpoolctl, plus the catalogue caches and saved draws. Bound sensitivity additionally needs the compatible torch/sbi model stack. The diagnostic CPU adapter deliberately remaps saved tensors to CPU; it does not remove the pinned-environment guard from the original scientific helpers.

The default paths are the repository's `research/` input archive and ignored `outputs/research_reanalysis/` for new outputs. `LG_PROJECT`, `LG_FINAL_ARCHIVE`, `LG_DIAGNOSTIC_ARTIFACTS` and `LG_SUPPLEMENT_OUTPUT` override those locations. Do not point the output variable at an original input archive.

The classifier artifact folder is `research/analysis_artifacts/classifier/`. It needs `verified_flamingo_frame.pkl`, `verified_tng_frame.pkl`, `domain_classifier_dynamics.pkl`, `domain_classifier_stellar_sum.pkl` and `audit_gate.json`. Import their exact source hashes using the repository artifact importer. The public `results/classifier/checkpoint_identities.csv` supplies checkpoint hashes. The original model registry, saved splits, evaluation draws and Local Group draws stay under `research/out/` and `research/data/`.

Set `LG_RUN_RESEARCH_WORKFLOW=1` only when ready. This flag also enables the research notebooks and the final-stage launcher. It is not required for the main result notebooks.

## Run in order

From the repository root, in PowerShell:

```powershell
$env:LG_RUN_RESEARCH_WORKFLOW = '1'
python research/supplements/prior/analyse_prior_shift.py
python research/supplements/prior/local_group_prior_swap.py
python research/supplements/prior/make_reporting_outputs.py
python research/supplements/bounds/run_sensitivity.py
python research/supplements/bounds/make_report.py
```

The first prior script regenerates the training-only KDE grids in the new prior output directory. The Local Group script uses those grids, normalizes within each observational component, then preserves the original component weights. The reporting script reads the outputs from those two steps. The bounds script validates recovered conditional distributions against archived validation losses and posterior draws before reporting the sensitivity. Each command reuses the original calculation; the changes here concern paths, explicit opt-in and output placement.

The optional classifier refit is separate:

```powershell
python research/supplements/diagnostic/support.py
```

It trains the original diagnostic domain classifier and validation calibrator, then reports saved-test calibration and local neighborhoods. It requires the copied audit gate. If the older group-OOF score archive is unavailable, it clearly skips only that historical comparison; the new classifier output is still reported. This does not train a new NPE or establish physical simulator evidence.

## Optional new final-stage fits on Roihu

`research/notebooks/final/` and `research/notebooks/diagnostics/` preserve the original Papermill parameter tags and expected launcher layout. The environment gate accepts the explicit flag above. The six-model and normalization workflows continue to validate their fixed inputs.

From `research/`, install the pinned `requirements-roihu.txt` in the documented CSC torch environment. Set `LG_RESEARCH_ENV` if using a separate environment and supply your own Slurm project account. A shell example is:

```bash
export LG_RUN_RESEARCH_WORKFLOW=1
export STAGE=smoke
sbatch --account=YOUR_PROJECT --export=ALL cluster/roihu/run_final_stage.sh
```

Use the new smoke receipt for subsequent prepare/train/evaluate stages. Preserve exact registry paths rather than choosing whichever run is newest. The human-readable design is `research/experiment_design.ipynb`; exact bytes of the historical design record are included as ignored `research/out/final_stage_audit/portability_v3/original_design.txt`, allowing the older checkpoint design hashes to remain unchanged.

The source files and original results are not deleted or replaced. New output timestamps and source hashes identify a new execution. A new training run or exact cross-hardware repetition is not implied by the existence of this documented workflow.
